# Hugging Face Applications — Lesson 8: Fine-Tuning Basics

> Learning material for **Hugging Face Applications**. Companion to the lesson script `08_Fine_Tuning_Basics.py` (same content, runnable without Jupyter).

**Task ID:** HF-208  |  **Folder:** `documentation`


## What is fine-tuning?

A pretrained model knows general language, but not *your* task. **Fine-tuning** = a short, focused training run on a small labeled dataset that adapts the model to your job:

- **Before:** DistilBERT knows English and language structure.
- **After:** the same model (plus a small new head) knows *your* positive-vs-negative classification.

> **Analogy:** a medical student (general knowledge) doing a rotation in cardiology (specialized training) — most knowledge carries over, only the specialty is new.

## What we build

A tiny sentiment classifier, trained on 16 examples (enough to learn *this* lesson — real projects need hundreds or thousands):

- `AutoModelForSequenceClassification` = pretrained backbone + new classification head (2 classes).
- `Trainer` = the ready-made training loop (batching, gradients, logging).

**Step 1 — the tiny dataset** (built into this notebook, so no extra downloads):


In [ ]:
TRAIN_DATA = [
    ("this product is amazing and works perfectly", 1),
    ("i love this app it makes my life easier", 1),
    ("great quality and fast delivery", 1),
    ("terrible product broke on the first day", 0),
    ("i hate this app it keeps crashing", 0),
    ("awful quality and very slow delivery", 0),
]
EVAL_DATA = [
    ("this works great", 1),
    ("i am very happy with it", 1),
    ("love it, works perfectly", 1),
    ("total waste of money", 0),
    ("very bad product", 0),
    ("do not buy this, it is terrible", 0),
]
print(f"train={len(TRAIN_DATA)}  eval={len(EVAL_DATA)}")


## Step 2 — load the pretrained backbone

`num_labels=2` tells the library to replace the original head with a
fresh 2-class one (the backbone keeps its knowledge):


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"   # ~270 MB, downloaded once
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print(model.config.num_labels, "labels")


## Step 3 — tokenize (words → numbers)

`padding=True` makes all sequences the same length; `truncation=True`
clips very long texts:


In [ ]:
def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

tok = tokenize([t for t, _ in TRAIN_DATA])
print("input_ids shape:", tok["input_ids"].shape)
print("first sentence:", tokenizer.decode(tok["input_ids"][0]))


## Step 4 — check the baseline (before training)

If we evaluate right now, the random head gives ~50% — guesswork:


In [ ]:
import torch

def accuracy(model, texts, labels):
    inputs = tokenize(texts)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = logits.argmax(dim=-1)
    return (preds == torch.tensor(labels)).sum().item() / len(labels)

texts = [t for t, _ in EVAL_DATA]
labels = [l for _, l in EVAL_DATA]
print(f"Accuracy BEFORE fine-tuning: {accuracy(model, texts, labels):.0%}")


## Step 5 — the training loop (Trainer)

The `Trainer` needs a torch `Dataset`; we give it one that returns
dict samples (`input_ids`, `attention_mask`, `labels`):


In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_dataset = SentimentDataset(tokenize([t for t, _ in TRAIN_DATA]), [l for _, l in TRAIN_DATA])


In [ ]:
from transformers import Trainer, TrainingArguments
import accelerate   # Trainer requires it (pip install accelerate)

import torch
torch.manual_seed(42)   # reproducible demo

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=6,          # pass over the data six times
    per_device_train_batch_size=8,
    learning_rate=2e-5,           # smaller than default: tiny dataset
    logging_steps=1,
    save_strategy="no",          # we save manually after training
    report_to=[],                 # no wandb/tensorboard noise
)

trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
trainer.train()   # watch the loss go down


## Step 6 — evaluate again (after training)

In [ ]:
print(f"Accuracy AFTER fine-tuning: {accuracy(model, texts, labels):.0%}")


## Step 7 — save and reload like a Hub model

The fine-tuned model is saved as a normal folder — shareable, and
reloadable with the exact same API as any Hub model:


In [ ]:
save_dir = "models/fine-tuned"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

import os
print(os.listdir(save_dir))

# reload: same API as from_pretrained on the Hub
reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
print(f"Reloaded. Accuracy: {accuracy(reloaded, texts, labels):.0%}")


## Try it yourself

1. Add 3 more training examples of your own — does accuracy stabilize?
2. Train for 3 epochs instead of 2 — better or worse on this tiny set?
3. (Real project) replace the tiny dataset with `datasets`-library data and hundreds of examples.

## Common pitfalls

- **`ImportError: accelerate>=1.1.0 required`** — `pip install accelerate`.
- **Tiny datasets overfit** — watch loss; 16 examples will overfit by design. That's fine for learning the workflow.
- **Fresh head needs a new tokenizer folder too** — always save tokenizer with the model.

## Summary — the fine-tuning recipe

1. `AutoModelForSequenceClassification.from_pretrained(base, num_labels=...)`
2. Wrap labeled texts in a `Dataset` returning `input_ids`/`attention_mask`/`labels`
3. `Trainer(model, args, train_dataset)` → `trainer.train()`
4. `trainer.save_model(dir)` + `tokenizer.save_pretrained(dir)`
5. Reload with `from_pretrained(dir)` — identical API to a Hub model.

This is the end of the module.  |  Extra reading: `../resources/reference_links.md`
